In [2]:
import json
import os
import numpy as np
from PIL import Image

## Resolve paths

In [12]:
def get_per_tile_base_path(input_channel_json_path, input_channel_json=None):
    if input_channel_json is None:
        with open(input_channel_json_path, 'r') as file:
            input_channel_json = json.load(file)
    stitching_dir = os.path.dirname(input_channel_json_path)
    flatfield_dir = os.path.join(stitching_dir, "per_tile_flatfields")
    channel_name = os.path.basename(os.path.dirname(input_channel_json[0]["file"]))
    channel_dir = os.path.join(flatfield_dir, channel_name)
    return channel_dir

def get_per_tile_folder_path(input_channel_json_path, tile_index, input_channel_json=None):
    if input_channel_json is None:
        with open(input_channel_json_path, 'r') as file:
            input_channel_json = json.load(file)
    total_tile_cnt = len(input_channel_json)
    assert 0 <= tile_index < total_tile_cnt, f"Tile index {tile_index} is out of range for {total_tile_cnt} tiles"
    channel_dir = get_per_tile_base_path(input_channel_json_path, input_channel_json)
    tile_dir = os.path.join(channel_dir, f"tile{tile_index}")
    return tile_dir

In [6]:
json_path = '/Volumes/data/sternsonlab/Zhenggang/2acq/outputs/M28C_LHA_S1/stitching/c0-n5.json'
with open(json_path, 'r') as file:
    json_file = json.load(file) # a list of dicts; each dict is a tile

In [9]:
print(f'{len(json_file)} tiles found')
print('Dictionary contents:')
for k, v in json_file[0].items():
    print(f'{k}: {v}')

20 tiles found
Dictionary contents:
type: GRAY16
index: 0
file: /data/sternsonlab/Zhenggang/2acq/outputs/M28C_LHA_S1/stitching/tiles.n5/c0/c0_M28C_LHA_S1.czi_tile0
position: [76.53478260869477, 39508.88695652174, -1516.0428571428574]
size: [1920, 1920, 1687]
pixelResolution: [0.23, 0.23, 0.42]


In [41]:
json_file[0]['size']

[1920, 1920, 1687]

In [11]:
get_per_tile_base_path(json_path)

'/Volumes/data/sternsonlab/Zhenggang/2acq/outputs/M28C_LHA_S1/stitching/per_tile_flatfields/c0'

In [13]:
get_per_tile_folder_path(json_path, 0)

'/Volumes/data/sternsonlab/Zhenggang/2acq/outputs/M28C_LHA_S1/stitching/per_tile_flatfields/c0/tile0'

## Load average baseline across all tiles

In [15]:
per_tile_base_path = get_per_tile_base_path(json_path)
avg_baseline_path = os.path.join(per_tile_base_path, "avg_baseline.npy")
avg_baseline = np.load(avg_baseline_path)
avg_baseline.shape

(1687,)

## Load df/ff/bn for each tile

In [16]:
tile0_path = get_per_tile_folder_path(json_path, 0)
all_fields = ['baseline', 'flatfield', 'darkfield']
tile0_fields = {field: np.load(os.path.join(tile0_path, f"{field}.npy")) for field in all_fields}
print(tile0_fields['baseline'].shape)
print(tile0_fields['flatfield'].shape)
print(tile0_fields['darkfield'].shape)


(1687,)
(1920, 1920)
(1920, 1920)


## Stack ff/df (2d) to 3d

In [17]:
S = 1 / tile0_fields['flatfield']

In [18]:
df_over_ff = tile0_fields['darkfield'] / tile0_fields['flatfield']

In [30]:
depth = tile0_fields['baseline'].shape[0]

def stack_2d_to_3d(arr_2d, z):
    return np.repeat(arr_2d[:, :, np.newaxis], z, axis=2)

df_over_ff_3d = stack_2d_to_3d(df_over_ff, depth)

In [20]:
df_over_ff_3d.shape


(1920, 1920, 1687)

In [22]:
arr_3d = np.arange(6).reshape(2, 3)
arr_3d

array([[0, 1, 2],
       [3, 4, 5]])

In [23]:
arr_3d = np.repeat(arr_3d[:, :, np.newaxis], 4, axis=2)
arr_3d

array([[[0, 0, 0, 0],
        [1, 1, 1, 1],
        [2, 2, 2, 2]],

       [[3, 3, 3, 3],
        [4, 4, 4, 4],
        [5, 5, 5, 5]]])

In [24]:
arr_3d[:, :, 0]


array([[0, 1, 2],
       [3, 4, 5]])

## Stack bn/avg_bn (1d) to 3d

In [31]:
def stack_1d_to_3d(arr_1d, x, y):
    return np.broadcast_to(arr_1d[np.newaxis, np.newaxis, :],  (x, y, len(arr_1d))) 

In [37]:
x, y = tile0_fields['flatfield'].shape
z = avg_baseline.shape[0]
avg_baseline_3d = stack_1d_to_3d(avg_baseline, x, y)
tile0_baseline_3d = stack_1d_to_3d(tile0_fields['baseline'], x, y)

In [38]:
print(avg_baseline_3d.shape)
print(tile0_baseline_3d.shape)
print(np.allclose(avg_baseline_3d[0, 0, :], avg_baseline))
print(np.allclose(tile0_baseline_3d[0, 0, :], tile0_fields['baseline']))

(1920, 1920, 1687)
(1920, 1920, 1687)
True
True


In [39]:
T = avg_baseline_3d - tile0_baseline_3d - df_over_ff_3d
T.shape

(1920, 1920, 1687)